# ClarifySQL · Notebook 05 — The QLoRA pilot

**Goal** (proposal, Section 9): "Pilot on about 100 examples and measure peak GPU memory, training tokens per
second and truncation." The pilot also checks, before the long runs, that

- the adapter learns the action format (valid JSON actions on the validation states),
- a saved adapter loads into the full agent and plays whole validation episodes,
- B3 is re-run with the corrected simulated user, so it can be compared with the fine-tuned model.

**Time:** about 35 minutes. **GPU quota:** about 35 minutes.
**Settings:** Kaggle → *Session options* → **GPU T4 x2**, **Internet On**. Colab → *T4 GPU*.

In [ ]:
!nvidia-smi

## 1. Get the project code

This cell clones the project from GitHub (branch `main`) into the notebook's working folder and makes
the `clarifysql` package importable. Running it again later just pulls the newest commits.

*Private repository?* Add a secret called `GITHUB_TOKEN` (Kaggle: **Add-ons → Secrets**; Colab: the
**key icon** in the left bar) and set `USE_TOKEN = True`. Our repository is public, so this is not needed.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Sathviktr/LMA_ClarifySQL.git"
BRANCH = "main"
USE_TOKEN = False  # only needed for a private repository

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{WORK}/LMA_ClarifySQL"

if USE_TOKEN:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    else:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{token}@")

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

sys.path.insert(0, f"{REPO}/src")
print("Running on:", "Kaggle" if ON_KAGGLE else "Colab / other", "| repo at", REPO)

In [ ]:
# Libraries that are not preinstalled (or too old) on Kaggle/Colab. PyTorch with CUDA is already there.
!pip install -q -r {REPO}/requirements-gpu.txt
!pip install -q -e {REPO} --no-deps  # lets the scripts import clarifysql
!pip install -q pytest

## 2. Truncation: do all training states fit in 1,024 tokens? (about 1 minute)

Every state is encoded exactly as it will be trained: the chat template, the prompt and the target action.
States longer than the limit would be left out (never cut), so this count is the "truncation" measure.

In [ ]:
from transformers import AutoTokenizer
from clarifysql.config import MODEL_ID, outputs_dir
from clarifysql.train.sft_data import encode_all, load_states

tok = AutoTokenizer.from_pretrained(MODEL_ID)
FULL_TOKENS = {}
for name in ("train", "validation", "train_noslot", "validation_noslot"):
    examples, over = encode_all(tok, load_states(name), max_len=1024)
    FULL_TOKENS[name] = sum(len(e.input_ids) for e in examples)
    print(f"{name:18} {len(examples)} states fit, {len(over)} longer than 1,024 tokens | "
          f"{FULL_TOKENS[name]:,} tokens per epoch")

## 3. The training code on a tiny random model (seconds)

The same loop that trains the real adapter, on a toy model on the CPU: the loss must fall, only the adapter
may change, and a saved adapter must reload and give the same answers.

In [ ]:
!cd {REPO} && python -m pytest -q tests/test_train.py tests/test_train_torch.py -p no:cacheprovider

## 4. The pilot: 100 training states, 2 epochs (about 12 minutes)

The base model is loaded in 4-bit NF4, a LoRA adapter (rank 16, attention + MLP) is attached and trained with
micro-batch 1 and 8 states per optimizer step. After each epoch the adapter is saved and writes its next action
for each of the 101 validation states.

In [ ]:
!python {REPO}/scripts/train_sft.py --name pilot --limit 100 --epochs 2 --grad-accum 8

## 5. What the pilot measured

In [ ]:
import json

log = json.load(open(f"{REPO}/results/train/pilot.json"))
print(f"{log['train_states']} states, {log['trainable_params']:,} trainable parameters, "
      f"{log['optimizer_steps']} optimizer steps")
for e in log["epochs"]:
    v = e["validation"]
    print(f"epoch {e['epoch']}: {e['seconds']:.0f} s, {e['tokens_per_second']:.0f} tokens/s, "
          f"peak {e['peak_allocated_gb']} GB | loss {e['train_loss']} (validation {e['val_loss']}) | "
          f"valid {v['valid_rate']}, decision F1 {v['decision_macro_f1']}, slots {v['slot_accuracy']}, "
          f"actions {v['action_accuracy']}")
minutes = FULL_TOKENS["train"] / log["epochs"][-1]["tokens_per_second"] / 60
print(f"\nOne full epoch ({FULL_TOKENS['train']:,} tokens) at this speed: about {minutes:.0f} minutes; "
      f"three epochs: about {3 * minutes:.0f} minutes")
print("loss per optimizer step:", [s["loss"] for s in log["steps"]])

In [ ]:
rows = [json.loads(line) for line in open(f"{REPO}/results/train/pilot_epoch-2_answers.jsonl")]
for r in rows[:6]:
    print("reference:", r["target"])
    print("pilot:    ", r["answer"], "\n")

## 6. Whole validation episodes (about 12 minutes)

The agent runs need the Spider databases. First B3 again, now with the corrected simulated user; then the
pilot adapter inside the full agent. The pilot saw only 100 states, so its scores are not the point: this
checks that a trained adapter loads and plays episodes before the long runs depend on it.

In [ ]:
os.environ["CLARIFYSQL_DATA"] = "/tmp/clarifysql_data"
!python {REPO}/scripts/download_data.py --only spider
ADAPTER = outputs_dir() / "adapters" / "pilot" / "epoch-2"
!python {REPO}/scripts/run_agent.py --system b3 --split validation
!python {REPO}/scripts/run_agent.py --system sft --split validation --adapter {ADAPTER} --name pilot_sft

## 7. Pack the results

In [ ]:
!cd {REPO} && zip -qr {WORK}/checkpoint5_results.zip results/train results/agent
!ls -la {WORK}/checkpoint5_results.zip

## 8. Send the results back

Download `checkpoint5_results.zip` (Kaggle: **Output** panel; Colab: **folder icon**) and unzip it **inside
the repository folder**, so the files land in `results/train/` and `results/agent/`. Then **stop the session**.